In [ ]:
# 0  imports
%load_ext autoreload
%autoreload 2
from afm_lib.instrument.session import call
from afm_lib.config import MODE_AC
from afm_lib.utils.recipe_builder import line_sites, grid_sites, check_sites, preview_sites, write_recipe
from afm_lib.schemas.recipe import load_recipe, site_program

async def stage_xy():
    raw = await call("get_experiment_status")
    return (raw["stage_x_m"], raw["stage_y_m"])

In [ ]:
# 1  sample corner — park the tip on the physical origin (e.g. lower-left electrode corner) by hand, then read it
CORNER = await stage_xy()
print(f"corner = ({CORNER[0]*1e3:+.4f}, {CORNER[1]*1e3:+.4f}) mm")

In [ ]:
# 2  direction — drive along the sample +x edge to a second point and read it
#    (or set DIRECTION = "x" / "y" if the sample edge is parallel to a stage axis)
DIRECTION = await stage_xy()
AXIS   = "y"          # the corner -> DIRECTION line is the sample +y edge
MIRROR = False
SAMPLE_FRAME = {"origin_stage_m": list(CORNER), "direction": list(DIRECTION), "axis": AXIS,
                "mirror": MIRROR, "note": "origin = lower-left corner; direction along the +y edge"}

In [ ]:
# 3a  P0 — drive to the first site, read it
P0 = await stage_xy()
print(f"P0 = ({P0[0]*1e3:+.4f}, {P0[1]*1e3:+.4f}) mm")

In [ ]:
# 3b  P1 — drive to the last site of the row, read it
P1 = await stage_xy()
span = ((P1[0]-P0[0])**2 + (P1[1]-P0[1])**2) ** 0.5
print(f"P1 = ({P1[0]*1e3:+.4f}, {P1[1]*1e3:+.4f}) mm   span = {span*1e3:.3f} mm")

In [ ]:
# 3c  grid, as in 07
N_SITES = 12
N_ROWS  = 1
ROW_SPACING_M = 4.0e-3

line  = line_sites(P0, P1, N_SITES)
sites = grid_sites(line, N_ROWS, ROW_SPACING_M)
print(f"{len(sites)} sites, spacing {span/(N_SITES-1)*1e3:.3f} mm")
for p in check_sites(sites):
    print("  !", p)
preview_sites(sites, sample_frame=SAMPLE_FRAME)    # left: stage mm; right: the same sites in sample mm, corner at the red +

In [ ]:
# 4  scan settings and recipe (the sample frame travels with the recipe; summary.py adds x/y_sample_m)
SCAN_SETTINGS = dict(x_scan_center_m=0.0, y_scan_center_m=0.0,      # offsets from the current frame centre
                     scan_size_m=1.0e-6, pixels=256, scan_rate_hz=1.0)

p = write_recipe("../recipes/pzh_02_AlBN_13%_topo.yaml",
                 name="pzh_02_AlBN_13%_topo",
                 sites=sites,
                 mode=MODE_AC,
                 scan_settings=SCAN_SETTINGS,
                 sample_frame=SAMPLE_FRAME,
                 context="AlBN 13% B, 200C, 56 min, 250nm — AC topography grid")
print("wrote", p)
print(p.read_text(encoding="utf-8")[:600])

In [ ]:
# 5  sanity and runtime
r = load_recipe(p); prog = site_program(r)
n_meas = len(r.sites) * len(prog)
per_frame = sum(m.scan_settings.pixels / m.scan_settings.scan_rate_hz + 10 for m in prog)
per_site  = 15 + 45                                    # MOVE_WAIT_S + APPROACH_WAIT_S
print(f"{r.name}: mode={r.mode!r}, {len(r.sites)} sites x {len(prog)} frame(s) = {n_meas} frames")
print(f"sample frame: {r.sample_frame}")
print(f"rough runtime: {(len(r.sites)*(per_site + per_frame))/3600:.1f} h")
print(f"recursion_limit to pass: {20 * n_meas + 100}")